# 2011 F1 practice → qualifying with a skew likelihood

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPy/f1_2011_skew_qualifying.ipynb)

**Draft** notebook: recreate the spirit of the 2011 University of Sheffield
qualifying-cutoff work in modern GPy, using curated 2011 timing via
`pods` and a **skew** observation model for non-pace laps.


## Origin of this work

In 2011 the University of Sheffield worked on predicting Formula 1
**qualifying cut-offs** from free-practice pace. The research code lived in
MATLAB under what we still call the *darwin* project
(`notebook/darwinMatlab/`): practice and qualifying lap tables fed a GP that
mapped practice times to qualifying times in the log domain,

\[
y = \log q - m\,\log p,
\]

with team and driver covariates and carefully tuned “consistency”
variances.

Lap-time noise is **not** symmetric. A car is more often *slower* than a
latent “true pace” (traffic, mistakes, cool-down) than faster. A Gaussian
observation model understates that right-hand tail. The O’Hagan / Azzalini
construction — a symmetric density multiplied by a probit — gives a proper
skewed density; that idea was used in the historical modelling and has now
been ported to GPy as `SkewGaussian` / `SkewExponential`
([GPy CIP-0014](https://github.com/SheffieldML/GPy)).

The 2011 timing tables (seventeen grands prix) are published as all-lap CSVs
on [`datasets_mirror`](https://github.com/lawrennd/datasets_mirror/tree/main/f1_2011_practice_qualifying)
and loaded by `pods.datasets.f1_2011_practice_qualifying`
([ods CIP-0005](https://github.com/lawrennd/ods)).

This notebook is a **draft** bridge between that history and the current
Python stack: load Silverstone 2011, show non-pace asymmetry, then compare a
Gaussian GP with a skew-likelihood GP for practice→qualifying prediction.


## Setup

Needs a GPy build that includes CIP-0014 (`GPy.likelihoods.SkewGaussian`) and
a pods build with CIP-0005. Editable installs from local clones work; otherwise
install from the relevant branches.


In [ ]:
import importlib
import subprocess
import sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        return importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return importlib.import_module(name)

np = _ensure("numpy")
pd = _ensure("pandas")
plt_mod = _ensure("matplotlib")
import matplotlib.pyplot as plt

try:
    import GPy
    assert hasattr(GPy.likelihoods, "SkewGaussian")
except (ImportError, AssertionError) as exc:
    raise ImportError(
        "Need GPy with SkewGaussian (CIP-0014). "
        "e.g. pip install -e /path/to/SheffieldML/GPy"
    ) from exc

try:
    import pods
    assert hasattr(pods.datasets, "f1_2011_practice_qualifying")
except (ImportError, AssertionError) as exc:
    raise ImportError(
        "Need pods with f1_2011_practice_qualifying (CIP-0005). "
        "e.g. pip install -e /path/to/lawrennd/ods"
    ) from exc

pods.access.overide_manual_authorize = True
GPy.plotting.change_plotting_library("matplotlib")
print("GPy OK; SkewGaussian available")
print("pods OK; f1_2011_practice_qualifying available")


## Load 2011 British Grand Prix (Silverstone)

Default demo race in the darwin demos. The loader returns:

- `X` — fastest clean practice lap (seconds), one row per driver  
- `Y` — Q1 / Q2 / Q3 bests (seconds)  
- `practice_laps` / `qualifying_laps` — **all** recorded laps (including pits)


In [ ]:
data = pods.datasets.f1_2011_practice_qualifying(
    grand_prix="GreatBritain",
    practice="FP3",
)

practice = data["practice_laps"].copy()
qualifying = data["qualifying_laps"].copy()
print(data["info"])
print("drivers with FP3+Q:", len(data["X"]))
print("FP3 lap rows:", len(practice), "  Q lap rows:", len(qualifying))
print("sample drivers:", list(data["abbreviation"][:6]))


## Non-pace times are right-skewed

For each driver, take clean FP3 laps and subtract that driver’s fastest clean
time. Residuals \(t - t_{\mathrm{best}}\) are mostly positive: traffic and
mistakes push you slow, while going *much* faster than your best is rare.
That is exactly the asymmetry a skew likelihood is meant to capture.


In [ ]:
clean = practice[(practice["pit_stop"].astype(int) == 0) & practice["lap_time_s"].notna()].copy()
# Flying-lap heuristic: discard out-laps / tours with absurd absolute times.
# Silverstone 2011 Q3 pole was ~90.4s; keep laps under 105s.
clean = clean[clean["lap_time_s"] < 105].copy()
best = clean.groupby("driver_number")["lap_time_s"].transform("min")
clean["delta_s"] = clean["lap_time_s"] - best

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(clean["delta_s"], bins=30, color="0.35", edgecolor="white")
ax.set_xlabel(r"FP3 flying lap $-$ driver best (s)")
ax.set_ylabel("count")
ax.set_title("Silverstone 2011 FP3: excess over personal best (flying laps)")
plt.show()

print("flying clean laps:", len(clean))
print("mean excess: {:.3f}s".format(clean["delta_s"].mean()))
print("fraction of flying laps > 0.5s off best: {:.1%}".format(
    (clean["delta_s"] > 0.5).mean()
))



## Darwin-style target in the log domain

Following `createGpModel`, work with

\[
y = \log q - m\,\log p
\]

for a fixed calibration slope \(m\) (here \(m=0.8\), as in
`regressGpPractice` for Q sessions). We use each driver’s **FP3 best** and
**Q3 best** (pole-order session). Drivers without a Q3 time are dropped.


In [ ]:
m_mean = 0.8  # consistency.mMean from regressGpPractice

p = data["X"].ravel()
q3 = data["Y"][:, 2]
abbr = np.asarray(data["abbreviation"])
team_index = np.asarray(data["team_index"], dtype=float)

mask = np.isfinite(p) & np.isfinite(q3) & (p > 0) & (q3 > 0)
p, q3, abbr, team_index = p[mask], q3[mask], abbr[mask], team_index[mask]

X = np.log(p)[:, None]
y = (np.log(q3) - m_mean * np.log(p))[:, None]

print("n drivers:", len(y))
print("FP3 best range: {:.3f} .. {:.3f} s".format(p.min(), p.max()))
print("Q3 best range:  {:.3f} .. {:.3f} s".format(q3.min(), q3.max()))


## Gaussian vs skew observation noise

Same kernel (RBF + linear on \(\log p\)), two likelihoods:

1. `Gaussian` — symmetric noise (baseline)  
2. `SkewGaussian` — O’Hagan skew-normal; positive \(\alpha\) puts more mass on
   \(y > f\) (slower / larger residual than the latent function), i.e. the
   non-pace direction

Inference for the skew model uses Laplace approximation (as in GPy CIP-0014).


In [ ]:
kern = GPy.kern.RBF(1, name="rbf") + GPy.kern.Linear(1, name="lin")

model_gauss = GPy.models.GPRegression(X, y, kernel=kern.copy())
model_gauss.optimize(messages=False, max_iters=200)

model_skew = GPy.core.GP(
    X,
    y,
    kern.copy(),
    GPy.likelihoods.SkewGaussian(variance=0.01, alpha=1.0),
    inference_method=GPy.inference.latent_function_inference.Laplace(),
)
model_skew.optimize(messages=False, max_iters=200)

print("Gaussian  log-likelihood: {:.3f}".format(float(np.asarray(model_gauss.log_likelihood()).reshape(()))))
print("Skew      log-likelihood: {:.3f}".format(float(np.asarray(model_skew.log_likelihood()).reshape(()))))
print("Skew alpha = {:.3f}  variance = {:.5f}".format(
    float(np.asarray(model_skew.likelihood.alpha).reshape(())),
    float(np.asarray(model_skew.likelihood.variance).reshape(())),
))


## Predictions back in seconds

Map latent predictions to qualifying time with
\(\hat q = \exp(\hat y + m\log p)\) and plot against actual Q3.


In [ ]:
def predict_q(model, X_log_p, p_seconds, m=m_mean):
    mu, var = model.predict(X_log_p)
    mu = np.asarray(mu).ravel()
    var = np.asarray(var).ravel()
    log_q = mu + m * np.log(p_seconds)
    q_hat = np.exp(log_q)
    # delta method-ish band on log_q then exp
    q_lo = np.exp(log_q - 2 * np.sqrt(var))
    q_hi = np.exp(log_q + 2 * np.sqrt(var))
    return q_hat, q_lo, q_hi

q_g, lo_g, hi_g = predict_q(model_gauss, X, p)
q_s, lo_s, hi_s = predict_q(model_skew, X, p)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, q_hat, lo, hi, title in [
    (axes[0], q_g, lo_g, hi_g, "Gaussian likelihood"),
    (axes[1], q_s, lo_s, hi_s, "SkewGaussian likelihood"),
]:
    ax.errorbar(q_hat, q3, xerr=[q_hat - lo, hi - q_hat], fmt="o", ms=4, color="0.2", alpha=0.8)
    lims = [min(q_hat.min(), q3.min()) - 0.2, max(q_hat.max(), q3.max()) + 0.2]
    ax.plot(lims, lims, ":", color="0.5")
    for a, x, yy in zip(abbr, q_hat, q3):
        ax.annotate(a, (x, yy), textcoords="offset points", xytext=(3, 3), fontsize=7)
    ax.set_xlabel("predicted Q3 (s)")
    ax.set_ylabel("actual Q3 (s)")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
fig.suptitle("Silverstone 2011: FP3 best → Q3 (darwin-style log model)")
fig.tight_layout()
plt.show()

rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
print("RMSE Gaussian: {:.3f} s".format(rmse(q_g, q3)))
print("RMSE Skew:     {:.3f} s".format(rmse(q_s, q3)))


## Optional: train on *all* clean FP3 laps

Using only the fastest lap throws away information. If we treat every clean
FP3 lap as an input that should predict the same driver’s Q3, non-pace laps
become right-skewed outliers around the latent mapping. The skew likelihood
is a natural noise model for that expanded training set.

(This is a draft experiment — not a full port of the MATLAB team/driver
index kernels.)


In [ ]:
# Map each clean FP3 lap to that driver's Q3 best
q3_by_driver = {int(d): float(t) for d, t in zip(data["driver_number"], data["Y"][:, 2])}
rows = []
for _, r in clean.iterrows():
    d = int(r["driver_number"])
    if d not in q3_by_driver or not np.isfinite(q3_by_driver[d]):
        continue
    pt, qt = float(r["lap_time_s"]), q3_by_driver[d]
    if pt <= 0 or qt <= 0:
        continue
    rows.append((np.log(pt), np.log(qt) - m_mean * np.log(pt), d))

Xa = np.array([[r[0]] for r in rows])
ya = np.array([[r[1]] for r in rows])
print("training points (all clean FP3 laps):", len(ya))

model_skew_all = GPy.core.GP(
    Xa,
    ya,
    GPy.kern.RBF(1) + GPy.kern.Linear(1),
    GPy.likelihoods.SkewGaussian(variance=0.05, alpha=1.5),
    inference_method=GPy.inference.latent_function_inference.Laplace(),
)
model_skew_all.optimize(messages=False, max_iters=200)
print("Skew-on-all-laps log-likelihood: {:.3f}".format(
    float(np.asarray(model_skew_all.log_likelihood()).reshape(()))
))
print("alpha = {:.3f}".format(float(np.asarray(model_skew_all.likelihood.alpha).reshape(()))))

# Predict at each driver's FP3 *best* for a fair compare
q_all, _, _ = predict_q(model_skew_all, X, p)
print("RMSE Skew (trained on all clean laps, eval at best): {:.3f} s".format(rmse(q_all, q3)))


## Next steps for this draft

- Port more of `createGpModel`: index kernels on team / driver, track bias,
  fixed-driver sampling flags from the MATLAB consistency structure.  
- Use Q1/Q2 windows (cut-off prediction) rather than only Q3.  
- Cross-validate across the other sixteen 2011 races in the pods pack.  
- Mirror a short GPc companion once the API is stable.

Related: `notebook/darwinMatlab/`, pods CIP-0005, GPy CIP-0014.
